In [5]:
!git clone https://github.com/vllm-project/speculators.git
%cd speculators

fatal: destination path 'speculators' already exists and is not an empty directory.
/content/speculators


In [7]:
# Speculators, data prep and training
!uv venv speculators
!uv pip install --python speculators/bin/python -e .

Using CPython 3.13.18 interpreter at: /usr/bin/python3
Creating virtual environment at: speculators
Activate with: source speculators/bin/activate
Using Python 3.13.18 environment at: speculators
Resolved 89 packages in 1.61s
Prepared 1 package in 510ms
Installed 89 packages in 551ms
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.16.0
 + attrs==26.1.0
 + certifi==2026.10.23
 + click==8.5.1
 + cuda-bindings==13.5.0
 + cuda-pathfinder==1.8.6
 + cuda-toolkit==13.0.3.0
 + datasets==5.1.0
 + dill==0.4.1
 + filelock==4.2.9
 + frozenlist==1.8.0
 + fsspec==2026.7.0
 + h11==0.16.0
 + hf-xet==1.8.0
 + hs-connectors==0.9.1a58 (from file:///content/speculators/hs_connectors)
 + httpcore==1.0.9
 + httpcore2==2.16.0
 + httpx==0.28.1
 + httpx2==2.16.0
 + huggingface-hub==2.4.0
 + idna==3.20
 + jinja2==3.1.6
 + jiter==0.19.0
 + loguru==0.7.3
 + markdown-it-py==4.2.0
 + markupsafe==3.0.4
 + mdurl==0.1.2
 + mpmath==1.3.0
 +

In [9]:
# vLLM, for serving the target model
!uv venv vllm
!uv pip install --python vllm/bin/python "vllm>=0.27.1"

Using CPython 3.13.18 interpreter at: /usr/bin/python3
Creating virtual environment at: vllm
Activate with: source vllm/bin/activate
Using Python 3.13.18 environment at: vllm
Resolved 199 packages in 83ms
Prepared 26 packages in 10m 18s
Installed 199 packages in 1.61s
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==2.6.0
 + anyio==4.16.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + attrs==26.1.0
 + blake3==1.0.11
 + cachetools==8.0.0
 + cbor2==6.2.1
 + certifi==2026.10.23
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.1
 + cloudpickle==3.1.2
 + compressed-tensors==0.17.0
 + cryptography==50.0.2
 + cuda-bindings==13.5.0
 + cuda-core==1.2.2
 + cuda-pathfinder==1.8.6
 + cuda-python==13.5.0
 + cuda-tile==1.8.0
 + cuda-toolkit==13.0.3.0
 + depyf==0.20.0
 + detect-installer==0.2.1
 + dill==0.4.1
 + dnspython==2.9.2
 + docstring-parser==0.18.0
 + einops==0.8.2
 + email-validat

In [10]:
%pip install -q datasets

In [11]:
from datasets import load_dataset
from collections import Counter
import json

# ejemplos por idioma
cuanto = {
    "es": 25,
    "ca": 25,
    "eu": 25,
    "gl": 25,
}



In [14]:
#Uso el dataset de SFT de ALIA, ya que incluye los idiomas que me interesan
dataset = load_dataset("BSC-LT/ALIA-2606-SFT",split="train",streaming=True).shuffle(seed=16, buffer_size=10000)

contar = Counter()

In [15]:

#Info relevante para recorrer el dataset en Dataset Structure en HF

with open("alia_prompts_pilot.jsonl", "w", encoding="utf-8") as file:
    for fila in dataset:
        language = fila["lang"]

        #At this moment we are only interested in es, ca, eu and gl
        if language not in cuanto:
            continue

        #We don't want to skip our limit
        if contar[language] >= cuanto[language]:
            continue

        messages = fila["messages"]

        json.dump({"id": fila["id"],"messages": messages}, file, ensure_ascii=False)
        file.write("\n")
        contar[language] += 1

        #We finish when we got everything we need
        if all(contar[l] >= n for l, n in cuanto.items()):
            break

print("Selected conversations:", dict(contar))

Selected conversations: {'es': 25, 'ca': 25, 'eu': 25, 'gl': 25}
